<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/v3/gpu-systems/fsdp/fsdp.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("fsdp", ...)`


# Implement FSDP (Fully Sharded Data Parallel) from Scratch

**Difficulty**: 🟣 Expert

**Companies**: Meta, Google, NVIDIA, Anthropic, xAI

---

### Problem Statement

Training large models requires distributing them across multiple GPUs. **Fully Sharded Data Parallel (FSDP)** is the state-of-the-art approach that shards model parameters, gradients, and optimizer states across all ranks (GPUs). This is based on Microsoft's **ZeRO** (Zero Redundancy Optimizer) paper.

The key insight: in standard Data Parallel (DDP), every GPU holds a **full copy** of the model. With FSDP:
- **At rest**: each rank holds only `1/world_size` of each parameter (a "shard")
- **Before forward**: parameters are **all-gathered** so each rank has the full parameter temporarily
- **After forward**: full parameters are discarded, keeping only the shard
- **During backward**: parameters are all-gathered again, because the gradient w.r.t. the layer's *input* (`dL/dx = dL/dy · W`) needs the full weight
- **After backward**: gradients are **reduce-scattered** (summed across ranks, then split) so each rank keeps the gradient for its shard only, and the optimizer updates only that shard

Like DDP this is **data parallelism**: every rank runs the whole model on its own slice of the global batch, so the per-rank weight gradients differ and their sum is the full-batch gradient. (Here `grad_output` is dL/dY of the *global* loss, so the per-rank pieces are summed. Real DDP/FSDP compute each rank's loss as a mean over its own micro-batch and then *average* the gradients across ranks, (1/W)·Σ_r ∇L_r, which is the same global-mean gradient when the micro-batches are equal in size.)

This reduces per-GPU memory for parameters, gradients and optimizer states from `O(model)` to `O(model / world_size)`. Activations are not sharded, and ZeRO-3 still needs one wrapped unit's full parameters transiently while that unit runs. The price is communication: two all-gathers and one reduce-scatter per step, about 1.5× the traffic of DDP's single all-reduce. With Adam in mixed precision, optimizer state (fp32 master weights plus two moments) is most of the memory, which is why sharding it (ZeRO-1/2) pays off first.

Since we cannot assume multi-GPU hardware, you will simulate distributed operations using a `FakeDistributed` class.

---

### Requirements

1. **FakeDistributed** — Simulate `world_size` ranks with `all_gather` and `reduce_scatter` on CPU.
2. **FSDPLinear** — A linear layer that shards its weight and bias, all-gathers them for the forward pass (`forward_on_rank`), and simulates a data-parallel backward plus reduce-scatter of the weight gradient (`compute_gradient_shards`).
3. **Three Strategies** — Full Shard (ZeRO-3), Shard Grad+Optimizer (ZeRO-2-like), and No Shard (DDP), with a per-rank memory report (`memory_per_rank`) covering parameters, gradients and Adam optimizer state.
4. **Validation** — Forward output must be identical regardless of sharding. Gathering the reduce-scattered gradient must give exactly the single-device gradient, and a sharded optimizer step must equal the single-device step.

---

### Constraints

- ✅ All computation happens on CPU using simulated distributed ops
- ✅ Must demonstrate correct gradient computation
- ✅ Must show per-rank memory usage for each strategy
- ❌ Do **not** use `torch.distributed` (we simulate everything)

---

<details>
  <summary>💡 Hint</summary>

  **Simulating distributed operations:**
  ```
  all_gather: Given shards [s0, s1, s2, s3], return full tensor = cat([s0, s1, s2, s3])
  reduce_scatter: Given full grads [g0, g1, g2, g3] (one per rank),
                  sum them, split the sum into chunks, rank i keeps chunk i
  ```
  
  **FSDP forward pass:**
  1. All-gather the parameter shards to reconstruct full weight
  2. Perform the forward computation with the full weight
  3. (Optionally) discard the full weight to save memory
  
  **FSDP backward pass:**
  1. All-gather parameters again (needed for dL/dx = dL/dy · W; dL/dW only needs the saved input)
  2. Each rank computes the full weight gradient from its OWN micro-batch
  3. Reduce-scatter gradients: each rank gets the summed gradient for its shard only
  4. Each rank updates only its shard (and keeps only its shard of the optimizer state)

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional
from enum import Enum
import math

In [ ]:
# Setup
torch.manual_seed(42)

# Simulated distributed configuration
WORLD_SIZE = 4  # simulate 4 GPUs

# Model dimensions
INPUT_DIM = 16
HIDDEN_DIM = 32
OUTPUT_DIM = 8
BATCH_SIZE = 4

print(f"Simulating {WORLD_SIZE} GPUs")
print(f"Model: {INPUT_DIM} -> {HIDDEN_DIM} -> {OUTPUT_DIM}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
class ShardingStrategy(Enum):
    """FSDP sharding strategies corresponding to ZeRO stages."""
    FULL_SHARD = "full_shard"        # ZeRO-3: shard params + grads + optimizer states
    SHARD_GRAD_OP = "shard_grad_op"  # ZeRO-2-like: shard grads + optimizer states; params
                                     # stay full through forward+backward
    NO_SHARD = "no_shard"            # DDP: replicate everything


class FakeDistributed:
    """
    Simulates torch.distributed operations on CPU.
    Instead of actual inter-GPU communication, we manipulate lists of tensors.
    """
    
    def __init__(self, world_size: int):
        self.world_size = world_size
    
    def shard(self, tensor: torch.Tensor, dim: int = 0) -> List[torch.Tensor]:
        """
        Split a tensor into world_size equal shards along dim.
        
        Args:
            tensor: full parameter tensor
            dim: dimension to shard along
        Returns:
            List of world_size shard tensors
        """
        # TODO: Split tensor into world_size chunks along dim
        # Use torch.chunk(tensor, self.world_size, dim=dim)
        # Return list of detached clones (each rank has its own copy)
        ...
    
    def all_gather(self, shards: List[torch.Tensor], dim: int = 0) -> torch.Tensor:
        """
        Simulate all-gather: concatenate shards from all ranks.
        In real distributed training, each rank sends its shard to all other ranks.
        
        Args:
            shards: list of tensors, one per rank
            dim: dimension to concatenate along
        Returns:
            Full tensor reconstructed from all shards
        """
        # TODO: Concatenate all shards along dim
        ...
    
    def reduce_scatter(self, full_grads: List[torch.Tensor], dim: int = 0) -> List[torch.Tensor]:
        """
        Simulate reduce-scatter: sum gradients across ranks, then scatter shards.
        Each rank ends up with the reduced gradient for its shard only.
        
        In real training:
        1. Each rank has the full gradient (computed from its local data)
        2. Gradients are summed across ranks (reduce)
        3. The summed gradient is split, each rank gets its shard (scatter)
        
        Args:
            full_grads: list of full gradient tensors, one per rank
            dim: dimension to shard along
        Returns:
            List of gradient shards, one per rank
        """
        # TODO: 
        # 1. Sum all full gradients element-wise: summed = sum(full_grads)
        # 2. Chunk the summed gradient into world_size shards
        # 3. Return list of shards (rank i gets chunk i)
        ...

print("FakeDistributed class defined.")

In [ ]:
class FSDPLinear(nn.Module):
    """
    A linear layer wrapped with FSDP-style sharding.
    
    The weight matrix is sharded across ranks along the output dimension.
    Before forward: all-gather to reconstruct the full weight.
    After backward: reduce-scatter the gradients.
    """
    
    def __init__(
        self,
        in_features: int,
        out_features: int,
        dist: FakeDistributed,
        strategy: ShardingStrategy = ShardingStrategy.FULL_SHARD,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.dist = dist
        self.strategy = strategy
        self.world_size = dist.world_size
        
        # TODO: Create the full weight and bias, then shard them
        # 1. Create full_weight of shape (out_features, in_features) with Xavier init
        # 2. Create full_bias of shape (out_features,)
        # 3. Shard the weight along dim=0 (output dimension)
        # 4. Shard the bias along dim=0
        # 5. Store shards as a list: self.weight_shards
        # 6. Store the full weight and bias as self.full_weight / self.full_bias
        #    (single-device reference) and the bias shards as self.bias_shards
        ...
    
    def forward_on_rank(self, x: torch.Tensor, rank: int) -> torch.Tensor:
        """
        Forward pass as seen by a specific rank.
        
        Args:
            x: input tensor (batch_size, in_features)
            rank: which simulated GPU is running this
        Returns:
            output: (batch_size, out_features)
        """
        # TODO:
        # 1. FULL_SHARD: all-gather the weight and bias shards to reconstruct the
        #    full parameters (the other strategies already hold them in full)
        # 2. Compute F.linear(x, full_weight, full_bias)
        # 3. FULL_SHARD drops the gathered copy right after use (del) - that release
        #    is what keeps per-rank parameter memory at 1/world_size
        ...
    
    def compute_gradient_shards(self, x: torch.Tensor, grad_output: torch.Tensor) -> List[torch.Tensor]:
        """
        Simulate the data-parallel weight-gradient computation and reduce-scatter.

        x and grad_output (= dL/dY of the global loss) cover the global batch.
        Rank r processes micro-batch r, computes the FULL weight gradient from it,
        and the reduce-scatter sums those per-rank gradients, so gathering the
        result gives exactly the single-device gradient grad_output.T @ x.

        Args:
            x: input tensor (batch_size, in_features)
            grad_output: gradient of loss w.r.t. output (batch_size, out_features)
        Returns:
            FULL_SHARD / SHARD_GRAD_OP: one weight-gradient shard per rank.
            NO_SHARD (DDP all-reduce): the full summed gradient on every rank.
        """
        # TODO:
        # 1. Split x and grad_output into one micro-batch per rank
        #    (torch.tensor_split always returns world_size pieces)
        # 2. Each rank computes its full weight gradient from its own micro-batch:
        #    shape (out_features, in_features) - mind the transpose order
        # 3. NO_SHARD: all-reduce (every rank gets the full sum);
        #    otherwise reduce-scatter to get per-rank gradient shards
        ...
    
    def memory_per_rank(self, optim_states_per_param: int = 2) -> dict:
        """
        Per-rank memory for this layer, in elements, during a training step:
        parameters as held from forward through backward, gradients after the
        reduction, and optimizer state (Adam keeps 2 states per
        parameter: exp_avg and exp_avg_sq).
        Returns a dict with 'params', 'grads', 'optimizer', 'total', 'strategy'.
        """
        # TODO: Compute memory based on strategy (shard = ceil(total / world_size))
        # FULL_SHARD:    params, grads and optimizer state all sharded
        # SHARD_GRAD_OP: params full (kept gathered from forward through backward);
        #                grads and optimizer state sharded
        # NO_SHARD:      everything replicated
        ...

print("FSDPLinear class defined.")

In [ ]:
# Validation
print("=" * 60)
print("VALIDATING FSDP IMPLEMENTATION")
print("=" * 60)

dist = FakeDistributed(world_size=WORLD_SIZE)

# Create test input
torch.manual_seed(42)
x = torch.randn(BATCH_SIZE, INPUT_DIM)

# Test 1: All-gather reconstructs the original tensor
print("\n--- Test 1: All-Gather Reconstruction ---")
test_tensor = torch.randn(HIDDEN_DIM, INPUT_DIM)
shards = dist.shard(test_tensor, dim=0)
print(f"Original shape: {test_tensor.shape}")
print(f"Shard shapes: {[s.shape for s in shards]}")
reconstructed = dist.all_gather(shards, dim=0)
print(f"Reconstructed shape: {reconstructed.shape}")
assert torch.equal(test_tensor, reconstructed), "All-gather reconstruction FAILED!"
assert sum(s.numel() for s in shards) == test_tensor.numel(), "Shards must be disjoint and cover the tensor!"
print("PASSED")

# Test 2: Reduce-scatter produces correct shards
print("\n--- Test 2: Reduce-Scatter ---")
per_rank = [torch.randn(HIDDEN_DIM, INPUT_DIM) for _ in range(WORLD_SIZE)]
result_shards = dist.reduce_scatter(per_rank, dim=0)
expected_shards = torch.chunk(torch.stack(per_rank).sum(0), WORLD_SIZE, dim=0)
for i, (got, expected) in enumerate(zip(result_shards, expected_shards)):
    assert torch.allclose(got, expected), f"Reduce-scatter shard {i} FAILED!"
print(f"Reduce-scatter produced {len(result_shards)} correct shards")
print("PASSED")

# Test 3: Forward pass produces same output regardless of rank
print("\n--- Test 3: Forward Pass Consistency ---")
layer = FSDPLinear(INPUT_DIM, HIDDEN_DIM, dist, ShardingStrategy.FULL_SHARD)
outputs = []
for rank in range(WORLD_SIZE):
    out = layer.forward_on_rank(x, rank)
    outputs.append(out)
    print(f"  Rank {rank} output shape: {out.shape}")

for i in range(1, WORLD_SIZE):
    assert torch.allclose(outputs[0], outputs[i], atol=1e-6), f"Rank {i} output differs!"
print("All ranks produce identical output. PASSED")

# Test 4: Forward matches single-device computation
print("\n--- Test 4: Correctness vs Single-Device ---")
single_output = F.linear(x, layer.full_weight, layer.full_bias)
fsdp_output = outputs[0]
max_err = (fsdp_output - single_output).abs().max().item()
print(f"Max error vs single-device: {max_err:.2e}")
assert torch.allclose(fsdp_output, single_output, atol=1e-6), "FSDP output != single-device!"
print("PASSED")

# Test 5: Data-parallel gradient + reduce-scatter == single-device gradient
print("\n--- Test 5: Gradient Reduce-Scatter ---")
grad_output = torch.randn(BATCH_SIZE, HIDDEN_DIM)
grad_shards = layer.compute_gradient_shards(x, grad_output)
print(f"Gradient shard shapes: {[s.shape for s in grad_shards]}")
full_grad = dist.all_gather(grad_shards, dim=0)
expected_grad = grad_output.T @ x  # single-device gradient of the same loss
max_err = (full_grad - expected_grad).abs().max().item()
print(f"Max error vs single-device gradient: {max_err:.2e}")
assert torch.allclose(full_grad, expected_grad, atol=1e-5), "Sharded gradient != single-device gradient!"
print("PASSED")

# Test 6: Sharded optimizer step == single-device step
print("\n--- Test 6: Sharded SGD Step ---")
lr = 0.1
new_shards = [w - lr * g for w, g in zip(layer.weight_shards, grad_shards)]  # each rank, own shard
stepped = dist.all_gather(new_shards, dim=0)
assert torch.allclose(stepped, layer.full_weight - lr * expected_grad, atol=1e-6), \
    "Updating each shard with its gradient shard must equal the single-device update!"
print("Each rank updated only its shard; gathered weights match the single-device step. PASSED")

# Test 7: Memory comparison across strategies
print("\n--- Test 7: Memory Comparison (elements per rank, Adam) ---")
print(f"{'Strategy':>15s} | {'Params':>7s} | {'Grads':>7s} | {'Optim':>7s} | {'Total':>7s} | {'vs DDP':>6s}")
print("-" * 66)
mems = {s: FSDPLinear(INPUT_DIM, HIDDEN_DIM, dist, s).memory_per_rank() for s in ShardingStrategy}
ddp_total = mems[ShardingStrategy.NO_SHARD]['total']
for strategy, mem in mems.items():
    print(f"  {strategy.value:>13s} | {mem['params']:>7d} | {mem['grads']:>7d} | {mem['optimizer']:>7d} "
          f"| {mem['total']:>7d} | {mem['total'] / ddp_total * 100:>5.0f}%")
assert (mems[ShardingStrategy.FULL_SHARD]['total'] < mems[ShardingStrategy.SHARD_GRAD_OP]['total']
        < mems[ShardingStrategy.NO_SHARD]['total']), "Expected ZeRO-3 < ZeRO-2 < DDP per-rank memory"
print("PASSED")

print("\nAll tests passed!")